# Market Cap Prompt Search

This notebook mirrors the `sector_prompt_search` pipeline for the `market_cap` metadata task.
Ground-truth labels are derived from `f_size` fetched via `DataFetcher`.

This version uses a 3-class setup:
- `small` (< 2B)
- `mid` (2B to 10B)
- `large` (>= 10B)


In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "data_collection" / "consts.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_analysis.data_fetcher.data_fetcher_class import DataFetcher


/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
market_cap_verbolizer = {
    "small": ["small", "smallcap", "micro", "tiny", "microcap"],
    "mid": ["mid", "medium", "med", "middle", "midcap"],
    "large": ["large", "big", "largecap", "mega", "giant", "megacap"],
}

market_cap_verbolizer = {
    "small": ["small"],
    "mid": ["mid"],
    "large": ["large"],
}

market_cap_verbolizer = {
    "small": ["small", "micro", "lower"],
    "mid": ["mid", "med", "middle"],
    "large": ["large", "big", "high"],
}

market_cap_prompts = [
    "Market cap bucket (small/mid/large):",
    "Classify market cap bucket. One-word output only: small|mid|large.",
    "Company size by market cap is: small/mid/large (one word).",
    "Estimate market-cap class from this filing. Output one word: small, mid, or large.",
    "If this issuer were sorted into size portfolios, the best market-cap class is: small, mid, or large.",
    "Infer the firm's size class using filing language only. Choose one: small, mid, large.",
    "Portfolio construction tag: assign a single market-cap class (small/mid/large). Tag:",
    "Read this report excerpt and pick the most likely market-cap bucket. Answer with one token: small, mid, or large.",
    "Ignore legal boilerplate and infer economic scale. Final market-cap class (small/mid/large):",
    "You are building a return-prediction dataset and need one size control label. Small - less than 2 billion; mid - from 2 to 10 billion; large - greater than 10 billion. Label:",
    "The market size of this company is:",
    "Rapid-fire classification for an equity screener. Output only the market-cap bucket (small/mid/large):",
    "Choose the issuer's market-cap size class from the allowed set {small, mid, large}. Output:",
    "I think the market size of this company is:", # Best one
    "If signals are mixed, choose the class most consistent with long-run investor perception of company size. Final class (small/mid/large):",
]



In [3]:
model_name = "mistralai/Mistral-7B-v0.1"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "12GiB", "cpu": "64GiB"},
)

model_driver = ModelDriver(model_name=model_name, model=model)


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.12it/s]
2026-06-04 19:47:50,856 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [4]:
MC_LABELS = ["small", "mid", "large"]
MC_BINS = [-float("inf"), 2e9, 1e10, float("inf")]


def bucket_market_cap(f_size: float) -> str | None:
    if pd.isna(f_size):
        return None
    return pd.cut(
        pd.Series([f_size]),
        bins=MC_BINS,
        labels=MC_LABELS,
        right=False,
    ).iloc[0]


fetcher = DataFetcher(DB_PARAMS)
reports_df = fetcher._fetch_reports(regressors=["raw_text", "f_size"])
reports_df = fetcher._apply_company_filters(reports_df, {})
reports_df = reports_df[reports_df["raw_text"].notna()].copy()
reports_df["market_cap_label"] = reports_df["f_size"].apply(bucket_market_cap)
reports_df = reports_df[reports_df["market_cap_label"].notna()].copy()
reports_df["market_cap_label"] = reports_df["market_cap_label"].astype(str)

label_counts = reports_df["market_cap_label"].value_counts().reindex(MC_LABELS, fill_value=0)
display(label_counts)

n_per_label = min(10, int(label_counts.min()))
if n_per_label < 1:
    raise ValueError("No samples available for at least one market-cap label.")

sample_parts = []
for label in MC_LABELS:
    label_df = reports_df[reports_df["market_cap_label"] == label]
    sample_parts.append(label_df.sample(n=n_per_label, random_state=42))

sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Rows per market-cap label in sample: {n_per_label}")
display(sample_df["market_cap_label"].value_counts().reindex(MC_LABELS, fill_value=0))
sample_df[["id", "f_size", "market_cap_label"]].head()


/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2

Available regressors:
 - avg_default_verbolizer
 - avg_shrink_verbolizer
 - doc_len
 - eps_surprise
 - f_size
 - full_list_default_verbolizer
 - full_list_shrink_verbolizer
 - hv_orig_score
 - lm_orig_score
 - max_abs_default
 - max_abs_shrink
 - max_default_verbolizer
 - max_shrink_verbolizer
 - md_hv1
 - md_hv2
 - md_hv3
 - md_lm1
 - md_lm2
 - md_lm3
 - min_default_verbolizer
 - min_shrink_verbolizer
 - stretch_default
 - stretch_shrink
Available sectors:
 - Technology (92)
 - Industrials (86)
 - Financial Services (85)
 - Healthcare (66)
 - Consumer Cyclical (58)
 - Consumer Defensive (40)
 - Real Estate (32)
 - Utilities (32)
 - Energy (30)
 - Basic Materials (23)
 - Communication Services (22)


/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:169: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  companies_df = pd.read_sql_query(query, conn)


market_cap_label
small      171
mid       1876
large    11468
Name: count, dtype: int64

Rows per market-cap label in sample: 10


market_cap_label
small    10
mid      10
large    10
Name: count, dtype: int64

,id,f_size,market_cap_label
0,57831,1.184793e+10,large
1,2667,9.455978e+09,mid
2,152966,2.988906e+10,large
3,9959,6.911583e+09,mid
4,91231,3.625553e+08,small


In [5]:
def evaluate_prompt(prompt: str, data: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    rows = []

    for row in tqdm(data.itertuples(index=False), total=len(data), desc=prompt[:48]):
        result = model_driver.predict_metadata(
            verbolizer=market_cap_verbolizer,
            prompt=prompt,
            text=row.raw_text,
        )
        probabilities = result["probabilities"]
        predicted_label = max(probabilities, key=probabilities.get)
        rows.append(
            {
                "id": row.id,
                "f_size": row.f_size,
                "true_label": row.market_cap_label,
                "predicted_label": predicted_label,
                "confidence": result["confidence"],
                "true_label_probability": probabilities[row.market_cap_label],
            }
        )

    predictions_df = pd.DataFrame(rows)
    y_true = predictions_df["true_label"]
    y_pred = predictions_df["predicted_label"]

    metrics = {
        "prompt": prompt,
        "avg_confidence": predictions_df["confidence"].mean(),
        "avg_true_label_probability": predictions_df["true_label_probability"].mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    return metrics, predictions_df


all_metrics = []
all_predictions = {}

for prompt in market_cap_prompts:
    metrics, predictions_df = evaluate_prompt(prompt, sample_df)
    all_metrics.append(metrics)
    all_predictions[prompt] = predictions_df

    print("\nPROMPT:", prompt)
    print(predictions_df["predicted_label"].value_counts())
    print(pd.crosstab(predictions_df["true_label"], predictions_df["predicted_label"]))

results_df = pd.DataFrame(all_metrics)
results_df = results_df.sort_values(["f1_macro", "avg_confidence"], ascending=[False, False]).reset_index(drop=True)
display(results_df)


Market cap bucket (small/mid/large):: 100%|██████████| 30/30 [01:19<00:00,  2.65s/it]



PROMPT: Market cap bucket (small/mid/large):
predicted_label
small    15
mid      11
large     4
Name: count, dtype: int64
predicted_label  large  mid  small
true_label                        
large                2    1      7
mid                  2    5      3
small                0    5      5


Classify market cap bucket. One-word output only: 100%|██████████| 30/30 [01:20<00:00,  2.67s/it]



PROMPT: Classify market cap bucket. One-word output only: small|mid|large.
predicted_label
small    26
mid       4
Name: count, dtype: int64
predicted_label  mid  small
true_label                 
large              0     10
mid                2      8
small              2      8


Company size by market cap is: small/mid/large (: 100%|██████████| 30/30 [01:16<00:00,  2.56s/it]


...

I think the market size of this company is:: 100%|██████████| 30/30 [01:22<00:00,  2.74s/it]



PROMPT: I think the market size of this company is:
predicted_label
mid      14
small    12
large     4
Name: count, dtype: int64
predicted_label  large  mid  small
true_label                        
large                2    5      3
mid                  1    7      2
small                1    2      7


If signals are mixed, choose the class most cons: 100%|██████████| 30/30 [01:19<00:00,  2.66s/it]


PROMPT: If signals are mixed, choose the class most consistent with long-run investor perception of company size. Final class (small/mid/large):
predicted_label
small    18
mid      11
large     1
Name: count, dtype: int64
predicted_label  large  mid  small
true_label                        
large                0    2      8
mid                  0    6      4
small                1    3      6


,prompt,avg_confidence,avg_true_label_probability,accuracy,precision_macro,recall_macro,f1_macro
0,I think the market size of this company is:,0.061659,0.367013,0.533333,0.527778,0.533333,0.501804
1,Market cap bucket (small/mid/large):,0.184641,0.359152,0.400000,0.429293,0.400000,0.387302
2,You are building a return-prediction dataset a...,0.015370,0.337901,0.366667,0.365190,0.366667,0.358006
3,The market size of this company is:,0.095916,0.375650,0.366667,0.439869,0.366667,0.350712
4,Company size by market cap is: small/mid/large...,0.019063,0.364989,0.433333,0.300000,0.433333,0.344444
5,"If signals are mixed, choose the class most co...",0.054954,0.402440,0.400000,0.292929,0.400000,0.333333
6,If this issuer were sorted into size portfolio...,0.010221,0.379892,0.400000,0.293651,0.400000,0.320191
7,Infer the firm's size class using filing langu...,0.027704,0.375631,0.400000,0.320000,0.400000,0.304762
8,Read this report excerpt and pick the most lik...,0.008299,0.371501,0.400000,0.320000,0.400000,0.304762
9,Estimate market-cap class from this filing. Ou...,0.014835,0.373379,0.366667,0.337607,0.366667,0.269231


<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>prompt</th>
      <th>avg_confidence</th>
      <th>avg_true_label_probability</th>
      <th>accuracy</th>
      <th>precision_macro</th>
      <th>recall_macro</th>
      <th>f1_macro</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>I think the market size of this company is:</td>
      <td>0.060412</td>
      <td>0.365226</td>
      <td>0.566667</td>
      <td>0.555556</td>
      <td>0.566667</td>
      <td>0.532107</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Company size by market cap is: small/mid/large...</td>
      <td>0.019013</td>
      <td>0.369207</td>
      <td>0.466667</td>
      <td>0.651852</td>
      <td>0.466667</td>
      <td>0.413822</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Market cap bucket (small/mid/large):</td>
      <td>0.185658</td>
      <td>0.360629</td>
      <td>0.400000</td>
      <td>0.477904</td>
      <td>0.400000</td>
      <td>0.389499</td>
    </tr>
    <tr>
      <th>3</th>
      <td>You are building a return-prediction dataset a...</td>
      <td>0.015264</td>
      <td>0.334761</td>
      <td>0.366667</td>
      <td>0.398990</td>
      <td>0.366667</td>
      <td>0.355556</td>
    </tr>
    <tr>
      <th>4</th>
      <td>If signals are mixed, choose the class most co...</td>
      <td>0.055400</td>
      <td>0.406570</td>
      <td>0.433333</td>
      <td>0.304625</td>
      <td>0.433333</td>
      <td>0.351396</td>
    </tr>
    <tr>
      <th>5</th>
      <td>If this issuer were sorted into size portfolio...</td>
      <td>0.009704</td>
      <td>0.380156</td>
      <td>0.433333</td>
      <td>0.326840</td>
      <td>0.433333</td>
      <td>0.344363</td>
    </tr>
    <tr>
      <th>6</th>
      <td>The market size of this company is:</td>
      <td>0.093217</td>
      <td>0.372084</td>
      <td>0.333333</td>
      <td>0.370833</td>
      <td>0.333333</td>
      <td>0.323443</td>
    </tr>
    <tr>
      <th>7</th>
      <td>Infer the firm's size class using filing langu...</td>
      <td>0.026929</td>
      <td>0.379960</td>
      <td>0.400000</td>
      <td>0.365385</td>
      <td>0.400000</td>
      <td>0.309524</td>
    </tr>
    <tr>
      <th>8</th>
      <td>Read this report excerpt and pick the most lik...</td>
      <td>0.008289</td>
      <td>0.373660</td>
      <td>0.400000</td>
      <td>0.365385</td>
      <td>0.400000</td>
      <td>0.309524</td>
    </tr>
    <tr>
      <th>9</th>
      <td>Classify market cap bucket. One-word output on...</td>
      <td>0.009695</td>
      <td>0.361696</td>
      <td>0.400000</td>
      <td>0.452381</td>
      <td>0.400000</td>
      <td>0.286550</td>
    </tr>
    <tr>
      <th>10</th>
      <td>Estimate market-cap class from this filing. Ou...</td>
      <td>0.015296</td>
      <td>0.374229</td>
      <td>0.366667</td>
      <td>0.337607</td>
      <td>0.366667</td>
      <td>0.269231</td>
    </tr>
    <tr>
      <th>11</th>
      <td>Rapid-fire classification for an equity screen...</td>
      <td>0.012201</td>
      <td>0.342286</td>
      <td>0.333333</td>
      <td>0.269231</td>
      <td>0.333333</td>
      <td>0.243386</td>
    </tr>
    <tr>
      <th>12</th>
      <td>Portfolio construction tag: assign a single ma...</td>
      <td>0.008139</td>
      <td>0.333271</td>
      <td>0.333333</td>
      <td>0.201058</td>
      <td>0.333333</td>
      <td>0.242218</td>
    </tr>
    <tr>
      <th>13</th>
      <td>Choose the issuer's market-cap size class from...</td>
      <td>0.006488</td>
      <td>0.336615</td>
      <td>0.300000</td>
      <td>0.173333</td>
      <td>0.300000</td>
      <td>0.196825</td>
    </tr>
    <tr>
      <th>14</th>
      <td>Ignore legal boilerplate and infer economic sc...</td>
      <td>0.088358</td>
      <td>0.357554</td>
      <td>0.333333</td>
      <td>0.123457</td>
      <td>0.333333</td>
      <td>0.180180</td>
    </tr>
  </tbody>
</table>
</div>

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>prompt</th>
      <th>avg_confidence</th>
      <th>avg_true_label_probability</th>
      <th>accuracy</th>
      <th>precision_macro</th>
      <th>recall_macro</th>
      <th>f1_macro</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Company size by market cap is: small/mid/large...</td>
      <td>0.015748</td>
      <td>0.384639</td>
      <td>0.466667</td>
      <td>0.651852</td>
      <td>0.466667</td>
      <td>0.413822</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Market cap bucket (small/mid/large):</td>
      <td>0.175093</td>
      <td>0.357509</td>
      <td>0.400000</td>
      <td>0.472222</td>
      <td>0.400000</td>
      <td>0.387413</td>
    </tr>
    <tr>
      <th>2</th>
      <td>If signals are mixed, choose the class most co...</td>
      <td>0.049566</td>
      <td>0.421211</td>
      <td>0.466667</td>
      <td>0.340351</td>
      <td>0.466667</td>
      <td>0.383908</td>
    </tr>
    <tr>
      <th>3</th>
      <td>The market size of this company is:</td>
      <td>0.066789</td>
      <td>0.367225</td>
      <td>0.400000</td>
      <td>0.477778</td>
      <td>0.400000</td>
      <td>0.381746</td>
    </tr>
    <tr>
      <th>4</th>
      <td>If this issuer were sorted into size portfolio...</td>
      <td>0.008810</td>
      <td>0.386758</td>
      <td>0.400000</td>
      <td>0.317460</td>
      <td>0.400000</td>
      <td>0.328906</td>
    </tr>
    <tr>
      <th>5</th>
      <td>Read this report excerpt and pick the most lik...</td>
      <td>0.007038</td>
      <td>0.387402</td>
      <td>0.400000</td>
      <td>0.365385</td>
      <td>0.400000</td>
      <td>0.309524</td>
    </tr>
    <tr>
      <th>6</th>
      <td>Infer the firm's size class using filing langu...</td>
      <td>0.024058</td>
      <td>0.390549</td>
      <td>0.400000</td>
      <td>0.320000</td>
      <td>0.400000</td>
      <td>0.304762</td>
    </tr>
    <tr>
      <th>7</th>
      <td>Classify market cap bucket. One-word output on...</td>
      <td>0.007302</td>
      <td>0.355701</td>
      <td>0.400000</td>
      <td>0.320000</td>
      <td>0.400000</td>
      <td>0.304762</td>
    </tr>
    <tr>
      <th>8</th>
      <td>Rapid-fire classification for an equity screen...</td>
      <td>0.010286</td>
      <td>0.347536</td>
      <td>0.366667</td>
      <td>0.282609</td>
      <td>0.366667</td>
      <td>0.286616</td>
    </tr>
    <tr>
      <th>9</th>
      <td>Estimate market-cap class from this filing. Ou...</td>
      <td>0.013929</td>
      <td>0.381006</td>
      <td>0.366667</td>
      <td>0.286667</td>
      <td>0.366667</td>
      <td>0.266667</td>
    </tr>
    <tr>
      <th>10</th>
      <td>Choose the issuer's market-cap size class from...</td>
      <td>0.004784</td>
      <td>0.360437</td>
      <td>0.333333</td>
      <td>0.233333</td>
      <td>0.333333</td>
      <td>0.266667</td>
    </tr>
    <tr>
      <th>11</th>
      <td>I think the market size of this company is:</td>
      <td>0.033342</td>
      <td>0.346399</td>
      <td>0.333333</td>
      <td>0.233918</td>
      <td>0.333333</td>
      <td>0.266183</td>
    </tr>
    <tr>
      <th>12</th>
      <td>You are building a return-prediction dataset a...</td>
      <td>0.012615</td>
      <td>0.331570</td>
      <td>0.266667</td>
      <td>0.169312</td>
      <td>0.266667</td>
      <td>0.199208</td>
    </tr>
    <tr>
      <th>13</th>
      <td>Ignore legal boilerplate and infer economic sc...</td>
      <td>0.084152</td>
      <td>0.359859</td>
      <td>0.333333</td>
      <td>0.123457</td>
      <td>0.333333</td>
      <td>0.180180</td>
    </tr>
    <tr>
      <th>14</th>
      <td>Portfolio construction tag: assign a single ma...</td>
      <td>0.006465</td>
      <td>0.335982</td>
      <td>0.333333</td>
      <td>0.111111</td>
      <td>0.333333</td>
      <td>0.166667</td>
    </tr>
  </tbody>
</table>
</div>

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>prompt</th>
      <th>avg_confidence</th>
      <th>avg_true_label_probability</th>
      <th>accuracy</th>
      <th>precision_macro</th>
      <th>recall_macro</th>
      <th>f1_macro</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>If signals are mixed, choose the class most co...</td>
      <td>0.054580</td>
      <td>0.406685</td>
      <td>0.466667</td>
      <td>0.324074</td>
      <td>0.466667</td>
      <td>0.378788</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Company size by market cap is: small/mid/large...</td>
      <td>0.018835</td>
      <td>0.369495</td>
      <td>0.433333</td>
      <td>0.300000</td>
      <td>0.433333</td>
      <td>0.344444</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Market cap bucket (small/mid/large):</td>
      <td>0.185007</td>
      <td>0.360145</td>
      <td>0.366667</td>
      <td>0.409722</td>
      <td>0.366667</td>
      <td>0.335276</td>
    </tr>
    <tr>
      <th>3</th>
      <td>If this issuer were sorted into size portfolio...</td>
      <td>0.010216</td>
      <td>0.376269</td>
      <td>0.400000</td>
      <td>0.293651</td>
      <td>0.400000</td>
      <td>0.320191</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Estimate market-cap class from this filing. Ou...</td>
      <td>0.014810</td>
      <td>0.374367</td>
      <td>0.400000</td>
      <td>0.370000</td>
      <td>0.400000</td>
      <td>0.314286</td>
    </tr>
    <tr>
      <th>5</th>
      <td>Read this report excerpt and pick the most lik...</td>
      <td>0.008224</td>
      <td>0.373681</td>
      <td>0.400000</td>
      <td>0.320000</td>
      <td>0.400000</td>
      <td>0.304762</td>
    </tr>
    <tr>
      <th>6</th>
      <td>Infer the firm's size class using filing langu...</td>
      <td>0.027592</td>
      <td>0.376534</td>
      <td>0.400000</td>
      <td>0.291667</td>
      <td>0.400000</td>
      <td>0.301471</td>
    </tr>
    <tr>
      <th>7</th>
      <td>Classify market cap bucket. One-word output on...</td>
      <td>0.009137</td>
      <td>0.355395</td>
      <td>0.366667</td>
      <td>0.306667</td>
      <td>0.366667</td>
      <td>0.285714</td>
    </tr>
    <tr>
      <th>8</th>
      <td>You are building a return-prediction dataset a...</td>
      <td>0.035152</td>
      <td>0.315203</td>
      <td>0.300000</td>
      <td>0.202381</td>
      <td>0.300000</td>
      <td>0.241453</td>
    </tr>
    <tr>
      <th>9</th>
      <td>Rapid-fire classification for an equity screen...</td>
      <td>0.011747</td>
      <td>0.341726</td>
      <td>0.333333</td>
      <td>0.240000</td>
      <td>0.333333</td>
      <td>0.241270</td>
    </tr>
    <tr>
      <th>10</th>
      <td>Choose the issuer's market-cap size class from...</td>
      <td>0.006455</td>
      <td>0.336563</td>
      <td>0.333333</td>
      <td>0.211180</td>
      <td>0.333333</td>
      <td>0.240048</td>
    </tr>
    <tr>
      <th>11</th>
      <td>Based on company scale cues in this filing, se...</td>
      <td>0.232406</td>
      <td>0.343879</td>
      <td>0.366667</td>
      <td>0.452381</td>
      <td>0.366667</td>
      <td>0.236045</td>
    </tr>
    <tr>
      <th>12</th>
      <td>Ignore legal boilerplate and infer economic sc...</td>
      <td>0.088316</td>
      <td>0.357191</td>
      <td>0.333333</td>
      <td>0.123457</td>
      <td>0.333333</td>
      <td>0.180180</td>
    </tr>
    <tr>
      <th>13</th>
      <td>Treat this as metadata annotation for firm siz...</td>
      <td>0.040183</td>
      <td>0.328895</td>
      <td>0.333333</td>
      <td>0.111111</td>
      <td>0.333333</td>
      <td>0.166667</td>
    </tr>
    <tr>
      <th>14</th>
      <td>Portfolio construction tag: assign a single ma...</td>
      <td>0.007956</td>
      <td>0.336096</td>
      <td>0.300000</td>
      <td>0.111111</td>
      <td>0.300000</td>
      <td>0.162162</td>
    </tr>
  </tbody>
</table>
</div>

In [6]:
top_3_prompts = results_df.head(3).copy()

display(top_3_prompts[["prompt", "avg_confidence", "precision_macro", "recall_macro", "f1_macro", "accuracy"]])

for idx, row in top_3_prompts.iterrows():
    print(f"\nTop {idx + 1} prompt")
    print(row["prompt"])
    print(
        {
            "avg_confidence": row["avg_confidence"],
            "precision_macro": row["precision_macro"],
            "recall_macro": row["recall_macro"],
            "f1_macro": row["f1_macro"],
            "accuracy": row["accuracy"],
        }
    )


,prompt,avg_confidence,precision_macro,recall_macro,f1_macro,accuracy
0,I think the market size of this company is:,0.061659,0.527778,0.533333,0.501804,0.533333
1,Market cap bucket (small/mid/large):,0.184641,0.429293,0.400000,0.387302,0.400000
2,You are building a return-prediction dataset a...,0.015370,0.365190,0.366667,0.358006,0.366667



Top 1 prompt
I think the market size of this company is:
{'avg_confidence': 0.0616590013106664, 'precision_macro': 0.5277777777777778, 'recall_macro': 0.5333333333333333, 'f1_macro': 0.5018037518037518, 'accuracy': 0.5333333333333333}

Top 2 prompt
Market cap bucket (small/mid/large):
{'avg_confidence': 0.18464111437400182, 'precision_macro': 0.4292929292929293, 'recall_macro': 0.39999999999999997, 'f1_macro': 0.38730158730158726, 'accuracy': 0.4}

Top 3 prompt
You are building a return-prediction dataset and need one size control label. Small - less than 2 billion; mid - from 2 to 10 billion; large - greater than 10 billion. Label:
{'avg_confidence': 0.015369938562313716, 'precision_macro': 0.3651903651903652, 'recall_macro': 0.3666666666666667, 'f1_macro': 0.35800552104899924, 'accuracy': 0.36666666666666664}


### Labeling note

`market_cap_label` is produced from `f_size` using 3-class market-cap thresholds (USD):
- `small`: `< 2B`
- `mid`: `2B - 10B`
- `large`: `>= 10B`
